# Chapter 3 (Hull) — Stock Index Futures (3.5) + Stack and Roll (3.6)

We continue Chapter 3 with two important practical topics:

## 3.5 Stock Index Futures
- What a stock index represents (a portfolio)
- How index futures are quoted (multiplier × index)
- How to hedge an equity portfolio with index futures
- How beta changes the hedge (not every portfolio moves like the index)

## 3.6 Stack and Roll
- What to do when the hedge horizon is longer than the maturity of available futures
- Rolling the hedge forward (close nearby contract, open a later one)
- Why roll results can be disappointing (futures below spot, imperfect compensation)

## 3.5 Stock Index Futures — Core ideas (Hull)

A stock index can be seen as the value of a **hypothetical portfolio of stocks**.

- The index level changes as the value of that portfolio changes.
- Dividends are usually **not** included in a “price index” (Hull notes exceptions like total return indices).

Index futures are standardized contracts.
They are usually quoted as:

$$
V_F = \text{Multiplier} \times \text{Index Level}
$$

Example multipliers (Hull-style):
- Mini S&P 500: $50 \times$ index
- Full S&P 500: $250 \times$ index
- Mini Nasdaq-100: $20 \times$ index

In [1]:
def futures_value(index_level, multiplier):
    return index_level * multiplier

index_level = 4010
multiplier = 50  # mini S&P 500 style
print("Futures contract value =", futures_value(index_level, multiplier))

Futures contract value = 200500


## Hedging an Equity Portfolio with Index Futures

Define:
- $V_A$ = current value of the portfolio
- $V_F$ = current value of one futures contract  
  (multiplier × index level)

### Case 1 — Portfolio mirrors the index (beta ≈ 1)

If the portfolio behaves like the index, Hull uses hedge ratio ≈ 1, and the number of futures contracts to short is:

$$
N^* = \frac{V_A}{V_F}
$$

(Short futures to hedge a long equity portfolio.)

In [2]:
V_A = 5_050_000                 # portfolio value
index_level = 1010
multiplier = 250
V_F = index_level * multiplier  # value of one futures contract

N_star = V_A / V_F
print(f"V_F = {V_F:,.0f}")
print(f"N*  = {N_star:.2f} -> rounded = {round(N_star)} contracts")

V_F = 252,500
N*  = 20.00 -> rounded = 20 contracts


### Case 2 — Portfolio has beta $\beta$

If the portfolio is more/less sensitive to the market than the index:

- $\beta > 1$: portfolio moves more than the index
- $\beta < 1$: portfolio moves less than the index

Hull’s beta-adjusted number of contracts:

$$
N^* = \beta \frac{V_A}{V_F}
$$

This is the practical “index hedge” formula.

In [3]:
beta = 1.5
N_star_beta = beta * V_A / V_F
print(f"N* (beta-adjusted) = {N_star_beta:.2f} -> rounded = {round(N_star_beta)} contracts")

N* (beta-adjusted) = 30.00 -> rounded = 30 contracts


## Changing the beta of a portfolio (not fully hedging to zero)

Sometimes you do not want to eliminate market exposure completely.
You may want to change beta from $\beta$ to a target $\beta'$.

Hull’s idea:
- If $\beta' < \beta$, you short futures (reduce market exposure)
- If $\beta' > \beta$, you go long futures (increase market exposure)

Number of contracts:

$$
N^* = (\beta - \beta') \frac{V_A}{V_F}
$$

- If $(\beta - \beta') > 0$ → short futures
- If $(\beta - \beta') < 0$ → long futures

In [4]:
beta = 1.5
beta_target = 0.75

N_change = (beta - beta_target) * V_A / V_F
print(f"Contracts to short to reduce beta: {N_change:.2f} -> rounded = {round(N_change)}")

Contracts to short to reduce beta: 15.00 -> rounded = 15


## Locking in the benefits of stock picking (Hull intuition)

If you believe your specific stock(s) will outperform the market,
you can hedge the market component with index futures:

- keep your stock/portfolio
- short index futures to remove broad market risk

Then performance depends mainly on whether your holdings beat the index
(“alpha” intuition), while market moves are largely hedged away.

## 3.6 Stack and Roll (Rolling the hedge forward)

Problem:
- Your hedge horizon is long (e.g., 1 year),
- but liquid futures contracts may only exist with nearer maturities.

Solution: **stack and roll**:
1. Short (or long) a nearby futures contract
2. Before it expires, close it out
3. Open a new position in a later maturity
4. Repeat until hedge horizon ends

This keeps the hedge in place through time.

In [5]:
import numpy as np

# Hull-style roll example (toy numbers inspired by the table)
# Each tuple: (sell_price_when_shorting, buy_price_when_closing)
rolls = [
    (48.20, 47.40),  # Oct 2021 contract
    (47.40, 46.50),  # Mar 2022 contract
    (46.30, 45.90),  # Jul 2022 contract
]

spot_start = 49.00
spot_end = 46.00

# Profit per barrel from each short futures: (sell - buy)
futures_profit = sum(sell - buy for sell, buy in rolls)
spot_drop = spot_start - spot_end

print(f"Spot drop:              {spot_drop:.2f} per barrel")
print(f"Total futures profit:   {futures_profit:.2f} per barrel")
print(f"Net effect (compared):  futures profit / spot drop = {futures_profit/spot_drop:.2%}")

Spot drop:              3.00 per barrel
Total futures profit:   2.10 per barrel
Net effect (compared):  futures profit / spot drop = 70.00%


**Interpretation (Hull):**
- The hedge is not guaranteed to offset the spot move perfectly.
- One common reason is that futures prices can be **below** expected future spot prices,
  so rolling may not fully “capture” the spot change.
- Liquidity and contract choice matter, and cashflow timing can also create issues.

## Summary

### Stock index futures
- Contract value: $V_F = \text{multiplier} \times \text{index level}$
- Full hedge (beta $\approx 1$): $N^* = \frac{V_A}{V_F}$
- Beta-adjusted hedge: $N^* = \beta\,\frac{V_A}{V_F}$
- Change beta to $\beta'$: $N^* = (\beta - \beta')\,\frac{V_A}{V_F}$  
  (short if positive, long if negative)

### Stack and roll
- When hedge horizon is longer than available liquid maturities, roll the position forward.
- Total hedge gains can differ from the spot move due to imperfect tracking and futures term structure.